In [ ]:
from mnist import Net, EuclideanNet, test

import matplotlib.pyplot as plt
from matplotlib import cm

import torch
from torchvision import datasets, transforms
import numpy as np
import os
import json
import dataclasses as dc

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")

np.random.seed(0)
torch.manual_seed(0);

In [ ]:
job_id = "436850"

dims = [2, 3]
args_paths = []
state_dict_paths = []
for dim in dims:
    args_path = f"checkpoints/{job_id}/fashion_args_{dim}d.json"
    args_paths.append(args_path)
    with open(args_path, "r") as f:
        args = json.load(f)
        state_dict_path = f"checkpoints/{job_id}/fashion_hyp_cnn_{args['best_epoch_hyp']}_{dim}d.pt"
        state_dict_paths.append(state_dict_path)

    assert os.path.isfile(state_dict_path)
    assert os.path.isfile(args_path)

data_path = "../fashion_data"
if os.getenv("GLOBALSCRATCH") is not None:
    data_path = os.path.join(os.getenv("GLOBALSCRATCH"), "fashion_data")

assert os.path.isdir(data_path)

In [ ]:
models_hyp = {}

for dim, args_path, state_dict_path in zip(dims, args_paths, state_dict_paths):
    with open(args_path, "r") as f:
        args = json.load(f)

    args = dc.make_dataclass("Args", [(k, type(v)) for k, v in args.items()])(**args)
    models_hyp[dim] = Net(args).to(device)

    state_dict = torch.load(state_dict_path, map_location=device)
    models_hyp[dim].load_state_dict(state_dict)

In [ ]:
args_paths = []
state_dict_paths = []
for dim in dims:
    args_path = f"checkpoints/{job_id}/fashion_args_{dim}d.json"
    args_paths.append(args_path)
    with open(args_path, "r") as f:
        args = json.load(f)
        state_dict_path = f"checkpoints/{job_id}/fashion_euc_cnn_{args['best_epoch_euc']}_{dim}d.pt"
        state_dict_paths.append(state_dict_path)

    assert os.path.isfile(state_dict_path)
    assert os.path.isfile(args_path)

In [ ]:
models_euc = {}

for dim, args_path, state_dict_path in zip(dims, args_paths, state_dict_paths):
    with open(args_path, "r") as f:
        args = json.load(f)

    args = dc.make_dataclass("Args", [(k, type(v)) for k, v in args.items()])(**args)
    models_euc[dim] = EuclideanNet(args).to(device)

    state_dict = torch.load(state_dict_path, map_location=device)
    models_euc[dim].load_state_dict(state_dict)

In [ ]:
data_path = "../fashion_data"
if os.getenv("GLOBALSCRATCH") is not None:
    data_path = os.path.join(os.getenv("GLOBALSCRATCH"), "fashion_data")

In [ ]:
save_imgs = True
if save_imgs:
    os.makedirs("exp_images", exist_ok=True)

In [ ]:
kwargs = {"num_workers": 1, "pin_memory": True} if device.type == "cuda" else {}
train_loader = torch.utils.data.DataLoader(
    datasets.FashionMNIST(
        data_path,
        train=True,
        download=True,
        transform=transforms.Compose(
            [transforms.ToTensor()]
        ),
    ),
    batch_size=args.batch_size,
    shuffle=True,
    **kwargs
)

test_loader = torch.utils.data.DataLoader(
    datasets.FashionMNIST(
        data_path,
        train=False,
        transform=transforms.Compose(
            [transforms.ToTensor()]
        ),
    ),
    batch_size=args.test_batch_size,
    shuffle=True,
    **kwargs
)

In [ ]:
# https://github.com/zalandoresearch/fashion-mnist
classes = {
    0: "T-shirt/top",
    1: "Trouser",
    2: "Pullover",
    3: "Dress",
    4: "Coat",
    5: "Sandal",
    6: "Shirt",
    7: "Sneaker",
    8: "Bag",
    9: "Ankle boot"
}

data, target = next(iter(test_loader))
data, target = data.to(device), target.to(device)

fig, ax = plt.subplots(2, 5, figsize=(10, 4))
colors = cm.rainbow(np.linspace(0, 1, 10))

for t in range(10):
    ax[t//5, t%5].scatter(0, 0, color=colors[t], s=50)

    idx = (target == t).nonzero()[0]
    ax[t//5, t%5].imshow(data[idx][0, 0].cpu(), cmap='gray')
    ax[t//5, t%5].set_title(classes[t])
    ax[t//5, t%5].axis('off')

plt.tight_layout()
if save_imgs:
    plt.savefig("exp_images/fashion_classes.png")
else:
    plt.show()

In [ ]:
test(None, models_hyp[2], device, test_loader);

In [ ]:
data, target = next(iter(test_loader))
data, target = data.to(device), target.to(device)
output = models_hyp[2](data)

num_samples = 60
sample_data = data.detach().cpu().numpy()[:num_samples]
sample_target = target.detach().cpu().numpy()[:num_samples]
sample_embeddings = models_hyp[3].embed(data)[:num_samples]

In [ ]:
np_sample_embeddings = sample_embeddings.detach().cpu().numpy()

fig, ax = plt.subplots(figsize=(10, 10))

circle = plt.Circle((0, 0), 1, color="black", fill=False)
ax.add_artist(circle)

colors = cm.rainbow(np.linspace(0, 1, 10))
for i, (x, y) in enumerate(np_sample_embeddings[:, :2]):
    img = sample_data[i, 0]
    ax.imshow(img, extent=(x+0.05-0.05, x+0.05+0.05, y+0.05-0.05, y+0.05+0.05), cmap="gray")
    ax.scatter(x, y, color=colors[sample_target[i]], s=50)

ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-1.1, 1.1)
ax.set_aspect("equal")

plt.axis("off")
if save_imgs:
    plt.savefig("exp_images/fashion_embedding_plot.png")
else:
    plt.show()

In [ ]:
x = np_sample_embeddings[:, 0]
y = np_sample_embeddings[:, 1]
z = np_sample_embeddings[:, 2]

r = np.sqrt(x ** 2 + y ** 2 + z ** 2)
theta = np.arccos(z / r)
s = np.array([1 if _y > 0 else -1 for _y in y])
phi = s * np.arccos(x / np.sqrt(x ** 2 + y ** 2))

spherical = np.zeros((r.size, 3))
spherical[:, 0] = r
spherical[:, 1] = theta
spherical[:, 2] = phi

In [ ]:
# We can try plotting the spherical coordinates

fig, ax = plt.subplots(figsize=(10, 10))

colors = cm.rainbow(np.linspace(0, 1, 10))
for i, (r, theta, phi) in enumerate(spherical):
    img = sample_data[i, 0]
    ax.scatter(phi, theta, color=colors[sample_target[i]], s=50, alpha=(1-r)**(0.25))

plt.xlabel("phi")
plt.ylabel("theta")
plt.show()

In [ ]:
# Export to visualization script

data, target = next(iter(test_loader))
data, target = data.to(device), target.to(device)

for dim in dims:
    np.save(f"embeddings_hyp_{dim}d.npy", models_hyp[dim].embed(data).detach().cpu().numpy())
    np.save(f"embeddings_euc_{dim}d.npy", models_euc[dim].embed(data).detach().cpu().numpy())

np.save(f"target.npy", target)